# 01 — Sayan Universal Gradient Theory: $\Theta_s$ Operator Basics

**Repository:** `sayan9168/Sayan-Universal-Gradient-Theory`  
**Zenodo DOI:** [10.5281/ZENODO.22723745](https://doi.org/10.5281/ZENODO.22723745)  
**Package:** `sayan-gradient`

---

## Overview

The **Sayan Universal Gradient (SUG)** framework couples released energy potential ($E_{nb}$), ambient thermal gradient ($\Delta T$), and distance ($r$) into a single continuous scalar field:

$$\theta_S(x) = K \, \frac{\sqrt{e}}{\tau \, \ln x}, \qquad x = \frac{r}{r_0} > 1$$

where the dimensionless ratios are defined with respect to reference scales:
$$e = \frac{E_{nb}}{E_0}, \qquad \tau = \frac{\Delta T}{T_0}, \qquad x = \frac{r}{r_0}$$

In this notebook, we explore:
1. Setting up reference scales (`ReferenceScales`).
2. Evaluating point values of $\theta_S(x)$ for scalar and vectorized inputs.
3. Examining the source core singularity at $x \le 1$.
4. Calculating the closed-form path-integrated field $\Theta_S(a \to b)$ using SciPy.
5. Computing the radial spatial gradient $\frac{d\theta_S}{dr}$.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sayan_gradient import (
    ReferenceScales,
    dimensionless,
    theta_s,
    legacy_theta_s,
    theta_s_line_integral,
    theta_s_gradient,
    ThetaSOperator,
    plot_field_decay,
)

## 1. Defining Reference Scales

Reference scales make the theory dimensionally consistent. For nuclear blast and laboratory regimes, unit scales are commonly used:

In [1]:
scales = ReferenceScales(E0=1.0, T0=1.0, r0=1.0, K=1.0)
print("Reference scales:", scales)

# Converting physical parameters to dimensionless state
E_nb = 50.0      # Energy units
delta_T = 273.0  # Temperature units
r = 1000.0       # Distance units

dim_state = dimensionless(E_nb, delta_T, r, scales=scales)
print(f"Dimensionless ratios: e = {dim_state.e:.2f}, tau = {dim_state.tau:.2f}, x = {dim_state.x:.2f}")

Reference scales: ReferenceScales(E0=1.0, T0=1.0, r0=1.0, K=1.0)
Dimensionless ratios: e = 50.00, tau = 273.00, x = 1000.00


## 2. Evaluating the $\Theta_s$ Field

We can evaluate the point field value using the modern dimensionless operator or the legacy base-10 scalar:

In [1]:
val_modern = theta_s(E_nb, delta_T, r, scales=scales)
val_legacy = legacy_theta_s(E_nb, delta_T, r)

print(f"Modern theta_S (natural log) = {float(val_modern):.6f}")
print(f"Legacy theta_S (base-10 log) = {val_legacy:.6f}")

Modern theta_S (natural log) = 0.003750
Legacy theta_S (base-10 log) = 0.008634


### Vectorized Field Evaluation

The operator supports full NumPy array broadcasting across distance arrays:

In [1]:
distances = np.array([2.0, 5.0, 10.0, 50.0, 100.0, 500.0, 1000.0])
field_values = theta_s(E_nb, delta_T, distances, scales=scales)

for d, f in zip(distances, field_values):
    print(f"Distance r = {d:6.1f} | theta_S = {f:.6f}")

Distance r =    2.0 | theta_S = 0.037368
Distance r =    5.0 | theta_S = 0.016093
Distance r =   10.0 | theta_S = 0.011249
Distance r =   50.0 | theta_S = 0.006621
Distance r =  100.0 | theta_S = 0.005624
Distance r =  500.0 | theta_S = 0.004168
Distance r = 1000.0 | theta_S = 0.003750


## 3. Singularity at Source Core ($x \le 1$)

Physical field descriptions break down inside the source core ($r \le r_0$). The package represents this naturally with `np.nan`:

In [1]:
r_core = np.array([0.5, 0.9, 1.0, 1.05, 2.0])
core_eval = theta_s(E_nb, delta_T, r_core, scales=scales)
print("Core evaluation results:")
for r_val, res in zip(r_core, core_eval):
    print(f"r = {r_val:4.2f} -> {res}")

Core evaluation results:
r = 0.50 -> nan
r = 0.90 -> nan
r = 1.00 -> nan
r = 1.05 -> 0.5308723128346884
r = 2.00 -> 0.037367745296585694


## 4. Radial Path-Integrated Operator (Contour Form)

The total field integrated along a radial path $[r_a, r_b]$ is:

$$\Theta_S(a \to b) = \frac{K\sqrt{e}}{\tau} \bigl[ \operatorname{li}(x_b) - \operatorname{li}(x_a) \bigr]$$

computed with SciPy's logarithmic integral via $\operatorname{Ei}(\ln x)$:

In [1]:
integral_exact = theta_s_line_integral(E_nb, delta_T, r_a=2.0, r_b=1000.0, scales=scales, method="exact")
integral_quad = theta_s_line_integral(E_nb, delta_T, r_a=2.0, r_b=1000.0, scales=scales, method="quad")

print(f"Contour integral (exact scipy expi) = {integral_exact:.6f}")
print(f"Contour integral (adaptive quad)   = {integral_quad:.6f}")

Contour integral (exact scipy expi) = 4.573258
Contour integral (adaptive quad)   = 4.573258


## 5. Radial Spatial Gradient

The spatial derivative $\frac{d\theta_S}{dr} = -\frac{\theta_S(r)}{r \ln(r/r_0)}$ describes the rate of field decay:

In [1]:
rad = 10.0
grad = theta_s_gradient(E_nb, delta_T, rad, scales=scales)
print(f"Spatial gradient d(theta_S)/dr at r={rad}: {grad:.6e} units^-1")

Spatial gradient d(theta_S)/dr at r=10.0: -4.885297e-04 units^-1


## 6. Visualization of Field Decay Curves

In [1]:
fig = plot_field_decay(
    energies=[0.5, 1.0, 2.0, 5.0, 10.0],
    tau=1.0,
    x_range=(1.1, 15.0),
    scales=scales,
    backend="matplotlib"
)
plt.show()